# ⚔️ ĐẤU TRƯỜNG CỜ VUA AI: ALPHAZERO (MCTS) vs DEEP Q-NETWORK (DQN)

Chào mừng bạn đến với **Đấu trường Cờ vua AI chuyên nghiệp**, một nền tảng thực nghiệm và trực quan hóa cuộc đối đầu đỉnh cao giữa hai trường phái Học tăng cường (Reinforcement Learning) hiện đại nhất hiện nay trên Google Colab.

Notebook này được thiết kế tự chứa (self-contained), tích hợp chính xác và khôi phục hoàn hảo hai cấu trúc mạng nơ-ron sâu thực tế từ mã nguồn dự án của bạn, đồng thời kết nối trực tiếp đến Google Drive của bạn để nạp các trọng số đã được huấn luyện.

---

## 📂 Hướng dẫn kết nối Google Drive & Đường dẫn trọng số

Để tiện lợi tối đa và không phải tải mô hình lên lại mỗi khi ngắt kết nối Google Colab, notebook này được lập trình để kết nối trực tiếp với **Google Drive** của bạn và tải các trọng số đã lưu trữ.

Hệ thống sẽ tự động tìm kiếm các trọng số tại:
*   Trọng số AlphaZero: `/content/drive/MyDrive/alphaOne/v1_finetuned_ep5.pth`
*   Trọng số DQN: `/content/drive/MyDrive/alphaOne/dqn_best.pth`

---

## 📚 1. CƠ SỞ LÝ THUYẾT & KIẾN TRÚC MÔ HÌNH: ALPHAZERO vs DQN

Dưới đây là bảng phân tích lý thuyết toán học chuyên sâu về cơ chế biểu diễn trạng thái, cách tính phần thưởng, cách quản lý điểm số hành động, và thuật toán học tập của cả hai mô hình.

---

### ♟️ A. BIỂU DIỄN TRẠNG THÁI BÀN CỜ (STATE REPRESENTATION - "BẢNG STATE")

Để các mạng nơ-ron tích chập sâu (CNN/ResNet) có thể hiểu được bàn cờ vua, trạng thái bàn cờ $s$ được mã hóa thành một **Tensor đặc trưng 3 chiều** kích thước **$(18, 8, 8)$** (thường được ví như "Bảng State" đa tầng):

*   **Lưới không gian ($8 \times 8$):** Đại diện cho 64 ô cờ từ dòng `1-8` và cột `a-h`.
*   **18 Kênh thông tin (18 Channels):**
    *   **Kênh 0 - 5:** Vị trí các quân cờ phía **Trắng** (Tốt, Mã, Tượng, Xe, Hậu, Vua). Ô có quân cờ tương ứng sẽ được điền giá trị `1.0`, các ô khác là `0.0`.
    *   **Kênh 6 - 11:** Vị trí các quân cờ phía **Đen** (Tốt, Mã, Tượng, Xe, Hậu, Vua).
    *   **Kênh 12:** Lượt đi hiện tại (toàn bộ lưới là `1.0` nếu là lượt Trắng, `0.0` nếu là lượt Đen).
    *   **Kênh 13 - 16:** Quyền nhập thành (Nhập thành gần/xa của Trắng, Nhập thành gần/xa của Đen - phủ `1.0` nếu còn quyền).
    *   **Kênh 17:** Vị trí ô bắt tốt qua đường (En-passant square) - điền `1.0` tại ô bắt tốt khả dĩ.

---

### 🏆 B. HÀM PHẦN THƯỞNG (REWARD FUNCTION)

Cách tính phần thưởng $R(s, a)$ khi chuyển từ trạng thái $s$ sang $s'$ qua hành động $a$ của hai mô hình phản ánh triết lý học tập khác nhau:

#### 1. Mô hình Deep Q-Network (DQN) — Dense Reward (Phần thưởng dày đặc)
DQN sử dụng một hệ thống phần thưởng dày đặc được phân bổ ở mỗi nước đi để AI nhanh chóng định hướng thế trận:
*   **Chiếu tướng (Checkmate):** $\mathbf{+10.0}$ (Thắng tuyệt đối)
*   **Hòa cờ (Draw Penalty):** $\mathbf{-0.2}$ (Phạt nhẹ để tránh lối chơi thụ động cầu hòa)
*   **Chênh lệch lực lượng ($\Delta$ Material Balance):** 
    $$R_{\text{material}} = \text{Material Balance}_{\text{sau}} - \text{Material Balance}_{\text{trước}}$$
    Với giá trị quân cờ chuẩn: Tốt $= 1.0$, Mã $= 3.0$, Tượng $= 3.0$, Xe $= 5.0$, Hậu $= 9.0$, Vua $= 0.0$.
*   **Hình phạt thời gian (Step Penalty):** $\mathbf{-0.01}$ mỗi nước đi để AI hướng tới sát cục nhanh nhất.
$$\text{Reward}_{\text{DQN}} = \Delta \text{Material} - 0.01 \quad (\text{hoặc } +10.0 / -0.2 \text{ khi kết thúc ván})$$

#### 2. Mô hình AlphaZero — Sparse Reward (Phần thưởng thưa)
AlphaZero thuần chất chỉ nhận phần thưởng khi **kết thúc ván đấu** ($z$):
*   **Thắng (Win):** $\mathbf{+1.0}$
*   **Thua (Loss):** $\mathbf{-1.0}$
*   **Hòa (Draw):** $\mathbf{-0.9}$ (Chiến thuật "Draw = Loss" để chống hòa thụ động trong cờ vua)
*   *Lưu ý:* Trong quá trình tìm kiếm, đầu ra **Value Head ($v$)** của mạng nơ-ron đóng vai trò như một **hàm lượng giá tự động** (learned dense surrogate), dự đoán trước giá trị kết cục $z$ từ các trạng thái chưa kết thúc để hướng dẫn cây MCTS.

---

### 📊 C. GIÁ TRỊ Q VÀ CÁCH QUẢN LÝ ĐIỂM SỐ HÀNH ĐỘNG (Q-VALUE & "BẢNG Q")

Giá trị Q đại diện cho điểm số chất lượng của hành động $a$ tại trạng thái $s$: $Q(s, a)$.

#### 1. Mô hình DQN — Ước lượng thông qua mạng nơ-ron (Function Approximation)
DQN **không sử dụng Bảng Q vật lý** vì bộ nhớ không thể chứa nổi $10^{120}$ thế cờ. Thay vào đó, mạng nơ-ron tích chập sâu (6 ResBlocks, 128 channels) đóng vai trò là bộ ước lượng hàm:
$$\text{Input: } s \text{ (18, 8, 8)} \xrightarrow{\text{ChessDQN}} \text{Output: Vector Q } \mathbf{q} \in \mathbb{R}^{4272}$$
*   Mỗi phần tử trong vector đầu ra đại diện trực tiếp cho giá trị $Q(s, a)$ của nước đi đó. AI đi quân bằng cách chọn nước đi hợp lệ có $Q(s, a)$ cao nhất:
    $$a^* = \arg\max_{a \in \text{Legals}} Q(s, a)$$

2. Mô hình AlphaZero — Duy trì động trong Cây tìm kiếm MCTS
AlphaZero **duy trì Bảng Q cục bộ** ngay trong các nút của cây tìm kiếm MCTS dưới dạng động:
*   Mỗi cạnh đi từ trạng thái $s$ qua nước đi $a$ sẽ lưu giữ:
    *   $N(s, a)$: Số lượt ghé thăm (Visit count)
    *   $W(s, a)$: Tổng giá trị tích lũy (Total action value) từ các nhánh lá truyền ngược về
    *   $Q(s, a)$: Giá trị trung bình của cạnh, được tính bằng:
        $$Q(s, a) = \frac{W(s, a)}{N(s, a)}$$
*   Giá trị $Q(s, a)$ này kết hợp với xác suất tiên nghiệm $P(s, a)$ (từ Policy Head) để dẫn đường cho công thức lựa chọn **PUCT**:
    $$a^* = \arg\max_{a} \left( Q(s, a) + C_{\text{puct}} \cdot P(s, a) \cdot \frac{\sqrt{N(s)}}{1 + N(s, a)} \right)$$

---

### 🧠 D. CƠ CHẾ HỌC TẬP VÀ CÔNG THỨC TOÁN HỌC (LEARNING MECHANISMS & FORMULAS)

#### 1. Thuật toán học tập của DQN: Học chênh lệch thời gian (Temporal Difference - TD)
DQN học bằng cách tối ưu hóa sai số TD giữa dự đoán hiện tại và mục tiêu tương lai thông qua **Target Network ($\theta^-$)** và **Experience Replay Buffer ($\mathcal{D}$)**.
*   **Mục tiêu huấn luyện (Negamax Target):** Vì cờ vua là trò chơi đối kháng 2 người tổng bằng không, nước đi tốt nhất tiếp theo của đối thủ chính là bất lợi của ta:
    $$Y^{\text{DQN}} = r - \gamma \max_{a' \in \text{Legals}(s')} Q(s', a'; \theta^-) \times (1 - \text{done})$$
*   **Hàm mất mát (Loss Function):** Sử dụng hàm Huber Loss (giúp chống bùng nổ gradient) để tối ưu hóa trọng số $\theta$:
    $$\mathcal{L}(\theta) = \mathbb{E}_{(s, a, r, s', d) \sim \mathcal{D}} \left[ \text{Huber}\left( Y^{\text{DQN}} - Q(s, a; \theta) \right) \right]$$

#### 2. Thuật toán học tập của AlphaZero: Học tự chơi dẫn đường bởi MCTS (Self-Play RL)
AlphaZero học bằng cách làm cho mạng nơ-ron dự đoán ngày càng khớp với kết quả tìm kiếm sâu sắc của MCTS và kết cục thực tế của ván cờ.
*   **MCTS làm bộ cải tiến chính sách (Policy Improvement Operator):** Ở mỗi trạng thái $s$, sau khi chạy MCTS, ta thu được phân phối xác suất tìm kiếm $\pi$ dựa trên số lượt ghé thăm:
    $$\pi_a = \frac{N(s, a)^{1/\tau}}{\sum_b N(s, b)^{1/\tau}}$$
*   **Hàm mất mát kết hợp (Dual-Head Loss Function):** Mạng nơ-ron $\theta$ được tối ưu hóa đồng thời cả đầu ra Policy ($\mathbf{p}$) và Value ($v$) theo công thức:
    $$\mathcal{L}(\theta) = (z - v)^2 - \pi^{\top} \log \mathbf{p} + c \|\theta\|^2$$
    Trong đó:
    *   $(z - v)^2$: Sai số bình phương trung bình (MSE) ép đầu ra Value $v$ dự đoán đúng kết quả ván đấu thực tế $z \in \{-1, 1\}$.
    *   $-\pi^{\top} \log \mathbf{p}$: Hàm mất mát Entropy chéo (Cross-Entropy) ép đầu ra Policy tiên nghiệm $\mathbf{p}$ trùng khớp với phân phối tìm kiếm thông thái $\pi$ của MCTS.
    *   $c \|\theta\|^2$: L2 regularization để tránh quá khớp (overfitting).

---

## 📊 E. BẢNG SO SÁNH ĐỐI LẬP HAI TRƯỜNG PHÁI AI

| Đặc trưng | 🔴 ALPHAZERO (MCTS) | ⚫ CHESSDQN |
| :--- | :--- | :--- |
| **Bản chất lý thuyết** | Model-Based Planning & Self-Play RL | Model-Free Temporal Difference (TD) |
| **Biểu diễn Q-value** | Lưu trữ động trong từng cạnh của cây MCTS | Ước lượng liên tục bằng vector đầu ra của CNN |
| **Hàm phần thưởng (Reward)**| Thưa (Sparse): Chỉ nhận $1.0 / -1.0$ khi hết ván | Dày đặc (Dense): Tính chênh lệch lực lượng mỗi nước |
| **Công thức mục tiêu** | Khớp chính sách $\pi$ của MCTS & Kết quả $z$ | Negamax TD-target: $r - \gamma \max Q(s'; \theta^-)$ |
| **Hàm Loss tối ưu** | Dual Loss: MSE (Value) + Cross-Entropy (Policy) | Huber Loss / MSE trên sai số TD |
| **Số khối ResNet** | **10 Blocks** (Kênh: 192) | **6 Blocks** (Kênh: 128) |
| **Tốc độ đi quân** | Chậm hơn, suy nghĩ sâu sắc (Deep thinking) | Cực nhanh, phản xạ tức thì (Instant reaction) |

---



## 1. Install Dependencies
We install `python-chess` for rules and visual SVG rendering in Google Colab.


In [ ]:
!pip install python-chess --quiet



## 2. Core Imports, Device & Google Drive Setup


In [ ]:
import os
import math
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import chess
from IPython.display import display, HTML, clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device in use: {device}")

# Mount Google Drive for persistent weights access
from google.colab import drive
drive.mount('/content/drive')



## 3. State & Action Encoder
Converts the chess board state into an `(18, 8, 8)` tensor and maps moves to indices `[0-4271]`.


In [ ]:
# Generate Action Space
FILES = "abcdefgh"
RANKS = "12345678"
PROMOTIONS = ['q', 'r', 'n', 'b']
ALL_UCI_MOVES = []

for f1 in FILES:
    for r1 in RANKS:
        for f2 in FILES:
            for r2 in RANKS:
                ALL_UCI_MOVES.append(f"{f1}{r1}{f2}{r2}")

for f1 in FILES:
    for f2 in FILES:
        if abs(FILES.index(f1) - FILES.index(f2)) <= 1:
            for p in PROMOTIONS:
                ALL_UCI_MOVES.append(f"{f1}7{f2}8{p}")
                ALL_UCI_MOVES.append(f"{f1}2{f2}1{p}")

ALL_UCI_MOVES = sorted(list(set(ALL_UCI_MOVES)))
MOVE_TO_INDEX = {move: i for i, move in enumerate(ALL_UCI_MOVES)}
INDEX_TO_MOVE = {i: move for move, i in MOVE_TO_INDEX.items()}

class Encoder:
    @staticmethod
    def board_to_tensor(board: chess.Board) -> torch.Tensor:
        """Converts chess board into an (18, 8, 8) tensor."""
        tensor = np.zeros((18, 8, 8), dtype=np.float32)
        
        # 1. Piece positions
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                color = 1 if piece.color == chess.WHITE else 0
                piece_type = piece.piece_type - 1 # 0 to 5
                channel = piece_type if color == 1 else piece_type + 6
                rank, file = chess.square_rank(square), chess.square_file(square)
                tensor[channel, rank, file] = 1.0
                
        # 2. Whose turn is it
        if board.turn == chess.WHITE:
            tensor[12, :, :] = 1.0
            
        # 3. Castling rights
        if board.has_kingside_castling_rights(chess.WHITE): tensor[13, :, :] = 1.0
        if board.has_queenside_castling_rights(chess.WHITE): tensor[14, :, :] = 1.0
        if board.has_kingside_castling_rights(chess.BLACK): tensor[15, :, :] = 1.0
        if board.has_queenside_castling_rights(chess.BLACK): tensor[16, :, :] = 1.0
        
        # 4. En-passant square
        if board.ep_square is not None:
            rank, file = chess.square_rank(board.ep_square), chess.square_file(board.ep_square)
            tensor[17, rank, file] = 1.0
            
        return torch.tensor(tensor)



## 4. AlphaZero Architecture (10 Blocks, 192 Channels)
This is your exact AlphaZeroNetV1 architecture matching `best_model.pth`.


In [ ]:
class ResBlockV1(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        return F.relu(out)

class AlphaZeroNetV1(nn.Module):
    def __init__(self, num_blocks: int = 10, num_channels: int = 192,
                 num_input_channels: int = 18, num_moves: int = 4272):
        super().__init__()

        self.initial_conv = nn.Sequential(
            nn.Conv2d(num_input_channels, num_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(num_channels),
            nn.ReLU(inplace=True)
        )

        self.res_blocks = nn.ModuleList([
            ResBlockV1(num_channels) for _ in range(num_blocks)
        ])

        # Policy Head
        self.policy_conv = nn.Conv2d(num_channels, 2, kernel_size=1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * 8 * 8, num_moves)

        # Value Head
        self.value_conv  = nn.Conv2d(num_channels, 1, kernel_size=1, bias=False)
        self.value_bn    = nn.BatchNorm2d(1)
        self.value_fc1   = nn.Linear(1 * 8 * 8, 256)
        self.value_fc2   = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor):
        x = self.initial_conv(x)
        for block in self.res_blocks:
            x = block(x)

        # Policy
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.reshape(p.size(0), -1)
        policy_logits = self.policy_fc(p)

        # Value
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.reshape(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))

        return policy_logits, value



## 5. Chess DQN Architecture (6 Blocks, 128 Channels)
This is your exact ChessDQN architecture matching `dqn_model.pth`.


In [ ]:
class ResBlockDQN(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        return F.relu(out)

class ChessDQN(nn.Module):
    def __init__(self, num_blocks: int = 6, num_channels: int = 128, 
                 num_input_channels: int = 18, num_moves: int = 4272):
        super().__init__()
        
        self.initial_conv = nn.Sequential(
            nn.Conv2d(num_input_channels, num_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(num_channels),
            nn.ReLU(inplace=True)
        )
        
        self.res_blocks = nn.ModuleList([
            ResBlockDQN(num_channels) for _ in range(num_blocks)
        ])
        
        # Q FC Head
        self.q_conv = nn.Conv2d(num_channels, 4, kernel_size=1, bias=False)
        self.q_bn   = nn.BatchNorm2d(4)
        
        policy_flat = 4 * 8 * 8
        self.q_fc1  = nn.Linear(policy_flat, 256)
        self.q_fc2  = nn.Linear(256, num_moves)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.initial_conv(x)
        for block in self.res_blocks:
            x = block(x)
            
        x_q = F.relu(self.q_bn(self.q_conv(x)))
        x_q = x_q.reshape(x_q.size(0), -1)
        x_q = F.relu(self.q_fc1(x_q))
        q_values = self.q_fc2(x_q)
        
        return q_values

    def select_action(self, state_tensor: torch.Tensor, legal_moves: list) -> tuple:
        """Selects action via masked Q-value calculation."""
        with torch.no_grad():
            q_values = self.forward(state_tensor)[0].cpu().numpy()
            
        masked_q = np.full(4272, -1e9, dtype=np.float32)
        for move in legal_moves:
            idx = MOVE_TO_INDEX.get(move.uci())
            if idx is not None:
                masked_q[idx] = q_values[idx]
                
        best_move_idx = np.argmax(masked_q)
        best_move_uci = INDEX_TO_MOVE[best_move_idx]
        return chess.Move.from_uci(best_move_uci), q_values[best_move_idx]



## 6. Weight Loading Utility


In [ ]:
def load_model_weights(model: nn.Module, filepath: str, model_name: str):
    """Loads state dict if the file exists, with smart nested key extraction."""
    if os.path.exists(filepath):
        try:
            checkpoint = torch.load(filepath, map_location=device)
            
            # Support both direct state dict and wrapped checkpoint dictionaries
            state_dict = None
            for key in ["model_state_dict", "state_dict"]:
                if isinstance(checkpoint, dict) and key in checkpoint:
                    state_dict = checkpoint[key]
                    break
            
            if state_dict is None:
                state_dict = checkpoint # Assume direct state_dict
                
            model.load_state_dict(state_dict)
            print(f"✅ Successfully loaded pre-trained weights for {model_name} from: '{filepath}'")
        except Exception as e:
            print(f"⚠️ Error loading weights for {model_name}: {e}. Initializing randomly.")
    else:
        print(f"⚠️ File '{filepath}' not found. {model_name} initialized with RANDOM weights.")



## 7. MCTS Node & AlphaZero Tree Search Guidance


In [ ]:
class MCTSNode:
    def __init__(self, prior=0.0):
        self.P = prior
        self.N = 0
        self.W = 0.0
        self.Q = 0.0
        self.children = {}

    def expand(self, legal_moves, action_probs, board=None):
        for move in legal_moves:
            idx = MOVE_TO_INDEX.get(move.uci(), 0)
            p = action_probs[idx]
            
            # Mate-in-1 priority check
            if board is not None:
                board.push(move)
                if board.is_checkmate():
                    p += 15.0
                board.pop()
                
            self.children[move] = MCTSNode(prior=p)

    def is_leaf(self):
        return len(self.children) == 0

    def select_child(self, c_puct=1.5):
        best_uct = -float('inf')
        best_child = None
        best_action = None
        sum_N = math.sqrt(self.N + 1)
        
        for action, child in self.children.items():
            u = c_puct * child.P * sum_N / (1 + child.N)
            uct = child.Q + u
            if uct > best_uct:
                best_uct = uct
                best_child = child
                best_action = action
        return best_action, best_child

    def backprop(self, value):
        self.N += 1
        self.W += value
        self.Q = self.W / self.N

def run_mcts_search(board: chess.Board, model: nn.Module, num_simulations=50) -> tuple:
    """Executes Monte Carlo Tree Search guided by AlphaZero policy/value outputs."""
    root = MCTSNode()
    
    # 1. Root Evaluation
    state_tensor = Encoder.board_to_tensor(board).unsqueeze(0).to(device)
    with torch.no_grad():
        logits, _ = model(state_tensor)
        probs = F.softmax(logits, dim=1)[0].cpu().numpy()
        
    root.expand(list(board.legal_moves), probs, board)
    
    # 2. Iterative Simulations
    for _ in range(num_simulations):
        sim_board = board.copy()
        node = root
        path = [node]
        
        # Selection
        while not node.is_leaf():
            action, child = node.select_child()
            path.append(child)
            sim_board.push(action)
            node = child
            
        # Evaluation & Expansion
        if sim_board.is_game_over():
            res = sim_board.result()
            # Draw punishment
            v = -0.9 if res == '1/2-1/2' else -1.0
        else:
            s_t = Encoder.board_to_tensor(sim_board).unsqueeze(0).to(device)
            with torch.no_grad():
                logits, value = model(s_t)
                probs = F.softmax(logits, dim=1)[0].cpu().numpy()
                v = value.item()
            node.expand(list(sim_board.legal_moves), probs, sim_board)
            
        # Backpropagation
        for n in reversed(path):
            v = -v
            n.backprop(v)
            
    # Selected action: Most visited node
    best_move = max(root.children.items(), key=lambda item: item[1].N)[0]
    return best_move, root



## 8. Initialize Models & Load Weights from Drive


In [ ]:
# 1. Instantiate the correct models
dqn_net = ChessDQN(num_blocks=6, num_channels=128).to(device)
az_net = AlphaZeroNetV1(num_blocks=10, num_channels=192).to(device)

dqn_net.eval()
az_net.eval()

print("Initializing structures... Please wait.")
time.sleep(1)

# 2. Load weights directly from mounted Google Drive
load_model_weights(az_net, "/content/drive/MyDrive/alphaOne/v1_finetuned_ep5.pth", "AlphaZero (MCTS)")
load_model_weights(dqn_net, "/content/drive/MyDrive/alphaOne/dqn_best.pth", "ChessDQN")



## 9. Automated Chess AI Battle Arena Loop
Runs the match in real time, visualizing the SVG board dynamically inside Google Colab.
You can customize who plays White ("mcts", "dqn", "random") and Black.


In [ ]:
def play_ai_battle(white_type="mcts", black_type="dqn", mcts_simulations=50, max_moves=100):
    """
    Simulates a head-to-head match between two chess AIs with real-time Colab rendering.
    Args:
        white_type: "mcts" (AlphaZero), "dqn", or "random"
        black_type: "mcts" (AlphaZero), "dqn", or "random"
        mcts_simulations: number of simulation rollouts per move
        max_moves: cap the match to prevent endless draws
    """
    board = chess.Board()
    move_count = 0
    
    print("🎬 Initializing Chess Arena... Round starts in 2 seconds.")
    time.sleep(2)
    
    while not board.is_game_over() and move_count < max_moves:
        turn = board.turn
        player_name = "🔴 WHITE (Trắng)" if turn == chess.WHITE else "⚫ BLACK (Đen)"
        agent_type = white_type if turn == chess.WHITE else black_type
        
        start_time = time.time()
        info = ""
        
        # 1. Agent selects move
        if agent_type == "mcts":
            move, root = run_mcts_search(board, az_net, num_simulations=mcts_simulations)
            info = f"AlphaZero MCTS (Sims: {mcts_simulations}, Root Visits: {root.N}, Eval Q: {root.Q:.4f})"
        elif agent_type == "dqn":
            state_tensor = Encoder.board_to_tensor(board).unsqueeze(0).to(device)
            move, q_val = dqn_net.select_action(state_tensor, list(board.legal_moves))
            info = f"ChessDQN (Estimated Q-value: {q_val:.4f})"
        else:
            move = random.choice(list(board.legal_moves))
            info = "Random Move (No model guidance)"
            
        elapsed = time.time() - start_time
        
        # 2. Push move to board state
        board.push(move)
        move_count += 1
        
        # 3. Dynamic graphic updates
        clear_output(wait=True)
        print("==========================================================================")
        print(f" ⚔️ CHESS AI ARENA | MOVE #{move_count} | PLAYER: {player_name}")
        print(f"  Current Algorithm: {agent_type.upper()}")
        print(f"  Decision Details : {info}")
        print(f"  Selected Action  : {move.uci()} (Calculated in {elapsed:.2f} seconds)")
        print("==========================================================================")
        
        # Render board SVG in Colab
        display(board)
        
        if board.is_check():
            print("🚨 CHECK! (Đang chiếu tướng!)")
            
        # Pause slightly to allow user to view the board state updates
        time.sleep(1.2)
        
    # Match conclusion
    clear_output(wait=True)
    print("========================= MATCH OVER =========================")
    if board.is_game_over():
        print(f"🏆 Final Result: {board.result()}")
        if board.is_checkmate():
            winner = "BLACK (Đen)" if board.turn == chess.WHITE else "WHITE (Trắng)"
            print(f"🎉 Checkmate! Winner: {winner}")
        elif board.is_stalemate() or board.is_insufficient_material() or board.is_seventyfive_moves() or board.is_fivefold_repetition():
            print("🤝 Draw (Hòa) by Chess Rules!")
    else:
        print(f"🛑 Match stopped by exceeding the move limit ({max_moves} moves).")
        
    display(board)



## 10. Start the Battle!
Run this cell to start the real-time face-off between your pre-trained models.
You can play with:
- `white_type=\"mcts\"` (AlphaZeroNetV1)
- `black_type=\"dqn\"` (ChessDQN)


In [ ]:
play_ai_battle(white_type="mcts", black_type="dqn", mcts_simulations=50, max_moves=80)
